# Physics 3926F — Lecture 5a Notebook
Python modules, linear algebra with NumPy/SciPy, and equilibrium (steady states) of ODEs

In [1]:
import numpy as np
import numpy.linalg as np_linalg
import scipy.linalg as sp_linalg

## 1. Writing your own module
A **module** is just a `.py` file. Your lab submissions (`LastnameFirstname_Lab5A.py`) are modules too.

The cell below uses `%%writefile` to save `fibo.py` next to this notebook, so the file really exists on disk.

In [2]:
%%writefile fibo.py
"""Fibonacci numbers module."""

def fib(n):
    """Print the Fibonacci series up to n."""
    a, b = 0, 1
    while a < n:
        print(a, end=' ')
        a, b = b, a + b
    print()

def fib2(n):
    """Return the Fibonacci series up to n as a list."""
    result = []
    a, b = 0, 1
    while a < n:
        result.append(a)
        a, b = b, a + b
    return result

if __name__ == "__main__":      # runs only with: python3 fibo.py
    print(fib2(10))

Overwriting fibo.py


In [3]:
import fibo

fibo.fib(42)               # module_name.function_name
result = fibo.fib2(42)
print(result)

0 1 1 2 3 5 8 13 21 34 
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


In [4]:
help(fibo.fib2)            # your docstring is the documentation

Help on function fib2 in module fibo:

fib2(n)
    Return the Fibonacci series up to n as a list.



### Activity: `import` vs. running the file
1. When we ran `import fibo` above, did the test code print anything?
2. Run the next cell, which executes the file as a script. What is different, and why?

In [5]:
!python3 fibo.py

[0, 1, 1, 2, 3, 5, 8]


`__name__` is `"__main__"` only when the file is run directly. When it is imported, `__name__` is `"fibo"`,
so test code inside the `if` block does not run on import. Use this for the test code in your lab files.

**Editing a module:** Python imports a module only once per session. After editing `fibo.py`, reload it:

In [6]:
import importlib
importlib.reload(fibo)
print(fibo.__name__)

fibo


## 2. Import styles
| Style | Example | Use? |
|---|---|---|
| Plain | `import math` | yes |
| Alias | `import numpy as np` | yes (standard convention) |
| Submodule alias | `import scipy.linalg as sp_linalg` | yes |
| Everything | `from numpy import *` | **avoid** |

In [7]:
import math as mt

testarr = np.zeros(5)
print("np.cos on an array:", np.cos(testarr))      # NumPy works element-wise on arrays
print("mt.cos on a scalar:", mt.cos(3.14))

try:
    mt.cos(testarr)                                 # math only takes scalars
except TypeError as err:
    print("mt.cos on an array -> TypeError:", err)

np.cos on an array: [1. 1. 1. 1. 1.]
mt.cos on a scalar: -0.9999987317275395
mt.cos on an array -> TypeError: only size-1 arrays can be converted to Python scalars


### Why `from numpy import *` is dangerous
It silently replaces Python's built-in functions that share a name (`sum`, `max`, `min`, `round`, ...).
We run it in a separate namespace here so this notebook stays clean.

In [8]:
print("built-in sum(range(5), -1):", sum(range(5), -1))     # sum(iterable, start) -> 10 + (-1)

ns = {}
exec("from numpy import *\nresult = sum(range(5), -1)", ns)  # now sum is np.sum(a, axis=-1)
print("after 'from numpy import *':", ns["result"])

built-in sum(range(5), -1): 9
after 'from numpy import *': 10


### Overlapping submodules
NumPy and SciPy both have `linalg`, `random` and `stats`. Import them under different names so it is always clear which one you call:
```python
import numpy.linalg as np_linalg
import scipy.linalg as sp_linalg
```

## 3. ODEs and Equilibrium (Steady States)
In Lecture 4 we solved ODEs by stepping forward in time. Now we ask a different question: **where does the system stop changing?**

- An **equilibrium** of an ODE is a state where the derivative is zero: $dx/dt = 0$.
- This is called the **equilibrium** or **steady-state** solution.
- For a **system** of ODEs, every derivative must be zero at the same time.
- Solving a **system of equations** finds the equilibrium points of the ODEs.

Whether the system actually *stays* at its equilibrium (stability) is the topic of Lecture 5b.

### Example: a system of three ODEs
$$\frac{dx}{dt} = x + 2y - z - 3$$
$$\frac{dy}{dt} = 2x - y + z - 1$$
$$\frac{dz}{dt} = x + y + z - 6$$

**Equilibrium:** set $dx/dt = 0$, $dy/dt = 0$ and $dz/dt = 0$:
$$x + 2y - z = 3,\qquad 2x - y + z = 1,\qquad x + y + z = 6$$

### The same thing as matrices
The whole ODE system is **one matrix equation**:

$$\frac{d}{dt}\begin{pmatrix} x \\ y \\ z \end{pmatrix}
= \underbrace{\begin{pmatrix} 1 & 2 & -1 \\ 2 & -1 & 1 \\ 1 & 1 & 1 \end{pmatrix}}_{A}
\begin{pmatrix} x \\ y \\ z \end{pmatrix}
- \underbrace{\begin{pmatrix} 3 \\ 1 \\ 6 \end{pmatrix}}_{\mathbf{b}}
\qquad\text{i.e.}\qquad \frac{d\mathbf{x}}{dt} = A\mathbf{x} - \mathbf{b}$$

At equilibrium $d\mathbf{x}/dt = 0$, so the equilibrium solves the linear system

$$A\mathbf{x} = \mathbf{b}$$

where $A$ is the **coefficient matrix**, $\mathbf{x}$ is the vector of **unknowns** and $\mathbf{b}$ is the **constant vector**.

**How to read off $A$ and $\mathbf{b}$:**
1. Put the unknowns in the **same order** in every row: here `x[0] = x`, `x[1] = y`, `x[2] = z`.
2. The coefficient of unknown *j* in equation *i* goes into `A[i, j]`. A missing unknown means a **0**.
3. The constant term of equation *i* goes into `b[i]` with its **sign flipped** (the ODE has $-3$, so $b_0 = +3$).

In [9]:
A = np.array([[1,  2, -1],
              [2, -1,  1],
              [1,  1,  1]], dtype=float)
b = np.array([3, 1, 6], dtype=float)     # the ODE constants were -3, -1, -6
print("A.shape =", A.shape, "  b.shape =", b.shape)

# Print the ODEs back from A and b, as a check on the bookkeeping
for i in range(A.shape[0]):
    terms = [f"{A[i, j]:+g} x[{j}]" for j in range(A.shape[1])]
    print(f"dx[{i}]/dt =", " ".join(terms), f"{-b[i]:+g}")

A.shape = (3, 3)   b.shape = (3,)
dx[0]/dt = +1 x[0] +2 x[1] -1 x[2] -3
dx[1]/dt = +2 x[0] -1 x[1] +1 x[2] -1
dx[2]/dt = +1 x[0] +1 x[1] +1 x[2] -6


### Activity (paper first, then check here)
Write this ODE system as $d\mathbf{x}/dt = A\mathbf{x} - \mathbf{b}$:
$$\frac{dx}{dt} = 3x - y + 2,\qquad \frac{dy}{dt} = x + 4y - 5$$

Type your $A$ and $\mathbf{b}$ below. The check compares them with the equations at a random point and prints `False` until your answer is right. Then find the equilibrium.

In [10]:
A2 = np.array([[0., 0.],
               [0., 0.]])                  # your answer
b2 = np.array([0., 0.])                    # your answer

def rhs2_by_hand(state):
    x, y = state
    return np.array([3*x - y + 2, x + 4*y - 5])

s2 = np.random.default_rng(0).random(2)    # any state will do
print("matches the ODEs:", np.allclose(np.dot(A2, s2) - b2, rhs2_by_hand(s2)))

matches the ODEs: False


## 4. Finding the equilibrium: solving $A\mathbf{x} = \mathbf{b}$

In [11]:
x = np_linalg.solve(A, b)
print("x, y, z =", x)

residual = np.dot(A,x) - b                 # Verify the solution by checking Ax - b = 0 (should be ~1e-16, not exactly 0)
#residual = A @ x - b                      # OR use this. @ is matrix multiplication (same as np.dot)
print("residual A x - b =", residual)      # should be ~1e-16, not exactly 0

x, y, z = [0.42857143 2.71428571 2.85714286]
residual A x - b = [ 0.0000000e+00  0.0000000e+00 -8.8817842e-16]


In [12]:
x_scipy = sp_linalg.solve(A, b)         # SciPy's solver
x_inv   = sp_linalg.inv(A).dot(b)       # works, but avoid: slower and less accurate
print(x_scipy)
print(x_inv)

[0.42857143 2.71428571 2.85714286]
[0.42857143 2.71428571 2.85714286]


## 5. Comparing two numerical results
Two methods rarely agree to the last digit. **Never test floats with `==`.** Ask instead: *how different are they, compared with what we should expect?*

In [13]:
print(0.1 + 0.2 == 0.3)                    # False!
print(0.1 + 0.2 - 0.3)

eps = np.finfo(float).eps                  # machine epsilon for 64-bit floats; this is the precision of your computer!
print("machine epsilon:", eps)

False
5.551115123125783e-17
machine epsilon: 2.220446049250313e-16


In [14]:
# Compare the solve() and inv() answers from Section 4
abs_diff = np.abs(x - x_inv)
rel_diff = abs_diff / np.abs(x)
print("absolute difference:", abs_diff)
print("relative difference:", rel_diff)
print("max relative difference in units of eps:", rel_diff.max() / eps)
print("np.allclose:", np.allclose(x, x_inv))

absolute difference: [2.22044605e-16 4.44089210e-16 0.00000000e+00]
relative difference: [5.18104078e-16 1.63611814e-16 0.00000000e+00]
max relative difference in units of eps: 2.333333333333333
np.allclose: True


A relative difference of a few × machine epsilon means the two answers agree to **rounding error**.
Report the size of the difference, not just "they are the same".

## Summary
- A module is a `.py` file. Import it, reload it after edits, and protect test code with `if __name__ == "__main__":`.
- Use `import numpy as np` style imports, and avoid `from module import *`.
- A linear ODE system can be written as $d\mathbf{x}/dt = A\mathbf{x} - \mathbf{b}$: coefficients into $A$ (0 if missing), constants into $\mathbf{b}$ with the sign flipped.
- Equilibrium (steady state): every derivative is zero, so solve $A\mathbf{x} = \mathbf{b}$.
- Solve with `solve`, not `inv`, and check the residual.
- Compare floats with relative differences, machine epsilon and `np.allclose`, never with `==`.